In [0]:
%pip install databricks-feature-engineering

In [0]:
dbutils.library.restartPython()

In [0]:
%sql

UPDATE credit_score.data.cadastral
SET CEP_2_DIG = NULL
WHERE LOWER(TRIM(CEP_2_DIG)) = 'na';


In [0]:
with open("fs_cadastro.sql", "r") as f:
    query = f.read()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

def table_exist(catalog, database, table):
    return spark.sql(
        f"SHOW TABLES FROM {catalog}.{database} LIKE '{table}'"
    ).count() > 0


dates = [
    '2018-08', '2018-09', '2018-10', '2018-11', '2018-12',
    '2019-01', '2019-02', '2019-03', '2019-04', '2019-05',
    '2019-06', '2019-07', '2019-08', '2019-09', '2019-10',
    '2019-11', '2019-12', '2020-01', '2020-02', '2020-03',
    '2020-04', '2020-05', '2020-06', '2020-07', '2020-08',
    '2020-09', '2020-10', '2020-11', '2020-12', '2021-01',
    '2021-02', '2021-03', '2021-04', '2021-05', '2021-06'
]

table_name = "feature_store.credit_score.fs_cadastral"

if table_exist("feature_store", "credit_score", "fs_cadastral"):

    for dt_ref in dates:
        print(f"Escrevendo {dt_ref}")

        df = spark.sql(
            query.format(dt_ref=f"{dt_ref}-01")
        )

        fe.write_table(
            name=table_name,
            df=df,
            mode="merge"
        )

    print("Table updated")

else:

    # cria usando a primeira snapshot
    first_date = dates[0]

    df = spark.sql(
        query.format(dt_ref=f"{first_date}-01")
    )

    fe.create_table(
        name=table_name,
        primary_keys=["ID_CLIENTE", "SAFRA_REF", "ID_DOCUMENTO"],
        partition_columns=["SAFRA_REF"],
        df=df
    )

    # adiciona as demais snapshots
    for dt_ref in dates[1:]:

        print(f"Escrevendo {dt_ref}")

        df = spark.sql(
            query.format(dt_ref=f"{dt_ref}-01")
        )

        fe.write_table(
            name=table_name,
            df=df,
            mode="merge"
        )

    print("Table created")